In [2]:
import gzip
import pandas as pd
import numpy as np
import sklearn
from sklearn import ensemble
from sklearn import tree
from sklearn import datasets
from sklearn.model_selection import train_test_split

In [3]:
print(sklearn.__version__)

1.0.2


In [4]:
def read_allc_index(allc_file):
    index_file = allc_file + ".idx"
    f = open(index_file,'r')
    chrom_pointer = {}
    for line in f:
        if line[0] != '#':
            fields = line.rstrip().split("\t")
            chrom_pointer[fields[0]] = int(fields[1])
    f.close()
    return(chrom_pointer)


In [3]:
def get_methylation_level_DMRfind(inputf_tsv,
                                  inputf_allc,
                                  output,
                                  min_cov=0,
                                  max_cov=None,
                                  buffer_line_number=100000,
                                  input_no_header=False):
    # open allc file
    allc_file = gzip.open(inputf_allc, "rt")
    # scan allc file to set up a table for fast look-up of lines belong
    # to different chromosomes
    chrom_pointer = read_allc_index(inputf_allc)

    # init
    prev_chrom = ""
    prev_end = ""

    # get methylation level for each DMR
    out = ""
    line_counts = 0
    g = output
    with open(inputf_tsv,'r') as f:
        if not input_no_header:
            f.readline() # skip header
        for line in f:
            line = line.rstrip("\n")
            fields = line.split("\t")
            dmr_chr=str(fields[0])
            dmr_start = int(fields[1])
            dmr_end = int(fields[2])

            # get to new chromosome and redirect pointer to related lines in allc file
            if prev_chrom != fields[0]:
                # if chrom is not in the allc file, add NAs
                if dmr_chr not in chrom_pointer:
                    out += line + "\t" + "\t".join(["NA" for _ in range(20)])+"\n"
                    line_counts += 1
                    if line_counts > buffer_line_number:
                        g.write(out)
                        line_counts = 0
                        out = ""
                    prev_chrom = dmr_chr
                    prev_end = dmr_end
                    continue
                else:
                    allc_file.seek(chrom_pointer[dmr_chr])
                    allc_prevbyte = chrom_pointer[dmr_chr]
                    # update allc line
                    allc_line=allc_file.readline()
                    allc_field=allc_line.split("\t")

            #If this new dmr overlaps with the previous, begin where the previous start was found
            elif prev_end and dmr_start < prev_end:
                allc_file.seek(allc_prevbyte)
                allc_line = allc_file.readline()
                allc_field = allc_line.split("\t")

            #read up to the beginning of the dmr
            byte = allc_prevbyte #in case the new dmr never enters the loop, keep byte the same as previously
            while allc_line and int(allc_field[1]) < dmr_start:
                byte = allc_file.tell()
                allc_line=allc_file.readline()
                allc_field=allc_line.split("\t")
            allc_prevbyte = byte #record the byte where dmr_start was found

            mc = 0
            h = 0
            mch = []
            while allc_line and int(allc_field[1]) >= dmr_start and int(allc_field[1]) <= dmr_end:
                h_site = int(allc_field[5])
                if h_site >= min_cov and (max_cov is None or h_site <= max_cov):
                    mc += int(allc_field[4])
                    h += h_site
                    mch.append(int(allc_field[4]) / h_site)
                allc_line=allc_file.readline()
                allc_field=allc_line.split("\t")
            if len(mch) > 0:
                counts = pd.cut(mch,bins=[-1, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1]).value_counts().to_list()
                methylation_level = [x/sum(counts) for x in counts]
                # methylation_level = str(float(mc) / h)
            else:
                methylation_level = ["NA" for _ in range(20)]

            out += line + "\t" + "\t".join([str(x) for x in methylation_level])+"\n"
            line_counts += 1
            if line_counts > buffer_line_number:
                g.write(out)
                line_counts = 0
                out = ""

            prev_chrom = dmr_chr
            prev_end = dmr_end

        if line_counts > 0:
            g.write(out)
            line_counts = 0

In [4]:
# fetal_Liver
file_path = "allCG_fetal_liver.tsv.gz"

output = "fetal_Liver.chr16.pmd.tsv"
input_tsv_file = "pmd.10kb.bed"

with open(output,'w') as g:
    # header
    g.write("chrom"+"\t"+"start"+"\t"+"end"+"\t"+
            "\t".join(["0.05", "0.1", "0.15", "0.2", "0.25",
                       "0.3", "0.35", "0.4", "0.45", "0.5", "0.55",
                       "0.6", "0.65", "0.7", "0.75", "0.8", "0.85",
                       "0.9", "0.95", "1"])+"\n")
    get_methylation_level_DMRfind(input_tsv_file,
                                  file_path,
                                  g,
                                  min_cov=5,
                                  max_cov=None,
                                  buffer_line_number=100000,
                                  input_no_header=True)

output = "fetal_Liver.chr16.non_pmd.tsv"
input_tsv_file = "non_pmd.10kb.bed"

with open(output,'w') as g:
    # header
    g.write("chrom"+"\t"+"start"+"\t"+"end"+"\t"+
            "\t".join(["0.05", "0.1", "0.15", "0.2", "0.25",
                       "0.3", "0.35", "0.4", "0.45", "0.5", "0.55",
                       "0.6", "0.65", "0.7", "0.75", "0.8", "0.85",
                       "0.9", "0.95", "1"])+"\n")
    get_methylation_level_DMRfind(input_tsv_file,
                                  file_path,
                                  g,
                                  min_cov=5,
                                  max_cov=None,
                                  buffer_line_number=100000,
                                  input_no_header=True)


output = "fetal_Liver.10kb.CG_percent.tsv"
input_tsv_file = "all_chr.10kb.bed"

with open(output,'w') as g:
    # header
    g.write("chrom"+"\t"+"start"+"\t"+"end"+"\t"+
            "\t".join(["0.05", "0.1", "0.15", "0.2", "0.25",
                       "0.3", "0.35", "0.4", "0.45", "0.5", "0.55",
                       "0.6", "0.65", "0.7", "0.75", "0.8", "0.85",
                       "0.9", "0.95", "1"])+"\n")
    get_methylation_level_DMRfind(input_tsv_file,
                                  file_path,
                                  g,
                                  min_cov=5,
                                  max_cov=None,
                                  buffer_line_number=100000,
                                  input_no_header=True)


pmd = pd.read_table("fetal_Liver.chr16.pmd.tsv", sep="\t").iloc[:, 3:].dropna(axis=0, how='any')
non_pmd = pd.read_table("fetal_Liver.chr16.non_pmd.tsv", sep="\t").iloc[:, 3:].dropna(axis=0, how='any')

X_data = pd.concat([pmd, non_pmd], axis=0).to_numpy()
y_data = np.array([1 for _ in range(pmd.shape[0])] + [0 for _ in range(non_pmd.shape[0])])
pmd

,0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95,1
0,0.000000,0.000000,0.076923,0.076923,0.307692,0.076923,0.076923,0.230769,0.000000,0.000000,0.076923,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.076923,0.000000,0.000000
1,0.833333,0.000000,0.000000,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.142857,0.057143,0.171429,0.200000,0.085714,0.057143,0.028571,0.028571,0.114286,0.028571,0.028571,0.028571,0.000000,0.000000,0.000000,0.028571,0.000000,0.000000,0.000000,0.000000
3,0.033333,0.022222,0.088889,0.066667,0.077778,0.088889,0.122222,0.100000,0.088889,0.111111,0.011111,0.077778,0.055556,0.011111,0.022222,0.022222,0.000000,0.000000,0.000000,0.000000
4,0.110092,0.055046,0.045872,0.091743,0.100917,0.064220,0.036697,0.119266,0.073394,0.045872,0.018349,0.082569,0.000000,0.091743,0.027523,0.000000,0.009174,0.027523,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,0.054152,0.007220,0.054152,0.064982,0.057762,0.043321,0.086643,0.101083,0.057762,0.101083,0.010830,0.086643,0.018051,0.068592,0.064982,0.046931,0.025271,0.028881,0.000000,0.021661
192,0.029851,0.004975,0.024876,0.079602,0.024876,0.054726,0.044776,0.124378,0.069652,0.144279,0.009950,0.189055,0.034826,0.059701,0.059701,0.014925,0.004975,0.019900,0.004975,0.000000
193,0.026178,0.010471,0.020942,0.099476,0.031414,0.041885,0.083770,0.104712,0.078534,0.198953,0.005236,0.109948,0.047120,0.031414,0.041885,0.020942,0.020942,0.010471,0.000000,0.015707
194,0.031250,0.018750,0.037500,0.043750,0.043750,0.062500,0.106250,0.087500,0.093750,0.137500,0.025000,0.131250,0.050000,0.062500,0.018750,0.037500,0.006250,0.000000,0.000000,0.006250


In [5]:
seed = 0
X_train,X_test,y_train,y_test = train_test_split(X_data,y_data,test_size = 0.3,
                                                 random_state=seed)

rfc = ensemble.RandomForestClassifier(
    n_estimators = 10000, max_features = None, oob_score = True, random_state=seed
)
rfc = rfc.fit(X_train,y_train)
rfc.score(X_test,y_test)

0.9693877551020408

In [6]:
fetal_Liver = pd.read_table("fetal_Liver.10kb.CG_percent.tsv").dropna(axis=0,how="any")
meta_data = fetal_Liver[["chrom", "start", "end"]]
predict_data = fetal_Liver.iloc[:, 3:]

res = rfc.predict(predict_data)
len(res[res == 1]) / len(res)

/tmp/ipykernel_44022/662256169.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  fetal_Liver = pd.read_table("./fetal_liver/allCG_fetal_Liver.10kb.CG_percent.tsv").dropna(axis=0,how="any")
/baimoc/wangjing/miniconda3/lib/python3.9/site-packages/sklearn/base.py:413: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


0.7383501596077773

In [7]:
res_tmp = meta_data[res == 1]
res_tmp.to_csv("res.bed", sep="\t", index=False, header=False)

In [8]:
fetal_Liver = pd.read_table("fetal_Liver.10kb.CG_percent.tsv")
na_regions = fetal_Liver[fetal_Liver.isna().any(axis=1)].iloc[:,:3]
na_regions.to_csv("na_regions.bed", sep="\t", index=False, header=False)

/tmp/ipykernel_44022/1851682945.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  fetal_Liver = pd.read_table("./fetal_liver/allCG_fetal_Liver.10kb.CG_percent.tsv")


In [11]:
file_path = "allCG_adult_Liver.tsv.gz"
output = "adult_Liver.10kb.CG_percent.tsv"
input_tsv_file = "all_chr.10kb.bed"

with open(output,'w') as g:
    # header
    g.write("chrom"+"\t"+"start"+"\t"+"end"+"\t"+
            "\t".join(["0.05", "0.1", "0.15", "0.2", "0.25",
                       "0.3", "0.35", "0.4", "0.45", "0.5", "0.55",
                       "0.6", "0.65", "0.7", "0.75", "0.8", "0.85",
                       "0.9", "0.95", "1"])+"\n")
    get_methylation_level_DMRfind(input_tsv_file,
                                  file_path,
                                  g,
                                  min_cov=5,
                                  max_cov=None,
                                  buffer_line_number=100000,
                                  input_no_header=True)

In [12]:
adult_Liver = pd.read_table("adult_Liver.10kb.CG_percent.tsv").dropna(axis=0,how="any")
meta_data = adult_Liver[["chrom", "start", "end"]]
predict_data = adult_Liver.iloc[:, 3:]

res_adult_liver = rfc.predict(predict_data)
len(res_adult_liver[res_adult_liver == 1]) / len(res_adult_liver)

/tmp/ipykernel_44022/2042457420.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  adult_Liver = pd.read_table("./adult_liver/allCG_adult_Liver.10kb.CG_percent.tsv").dropna(axis=0,how="any")
/baimoc/wangjing/miniconda3/lib/python3.9/site-packages/sklearn/base.py:413: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


0.07662786967877927

In [14]:
res_tmp = meta_data[res_adult_liver == 1]
res_tmp.to_csv("res_adult_liver.bed", sep="\t", index=False, header=False)